# Prepare 2024 flight data

Use the shared functions in `src/prepare_flights.py`. Airport selection remains fixed from 2024. Raw files are never edited. Weather matching is a separate step.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.prepare_flights import (
    COLUMNS, load_flights, prepare_timestamps, save_flights,
)

YEAR = 2024

## Load and validate flights

Read exactly twelve monthly files. Retain the selected origins, exclude cancelled flights and missing labels, then check dates, duplicate flight identifiers, and label consistency.

In [ ]:
flights, audit = load_flights(PROJECT_ROOT, YEAR)
missingness = flights[COLUMNS].isna().sum().rename("missing")

display(audit)
display(missingness)
print(f"Retained before timestamp checks: {len(flights):,} flights")

## Prepare UTC timestamps

Convert airport-local scheduled departures to UTC. A recorded time of 2400 means the following midnight. Exclude ambiguous or nonexistent daylight-saving times and retain them in an audit. Calculate prediction cutoffs two elapsed hours before departure.

In [ ]:
flights, excluded, timestamp_audit = prepare_timestamps(flights)

display(timestamp_audit)
display(excluded[["FlightDate", "Origin", "Dest", "CRSDepTime",
                  "scheduled_departure_local", "timezone"]])

## Save flight dataset and audits

The missingness audit describes retained flights before timestamp exclusions. Monthly cleaning counts and timestamp exclusions are recorded separately. The historical 2024 `excluded_ambiguous_departures.csv` is preserved; new runs use `excluded_unresolved_departures.csv` to cover both ambiguous and nonexistent times.

In [ ]:
save_flights(
    PROJECT_ROOT, YEAR, flights, audit, missingness, excluded, timestamp_audit
)
print(f"Saved {len(flights):,} flights for {YEAR}")